# Citation Prediction
## Team CiteSeeing: Athanasios Tonios, Panagiotis Trypos

The code implements a link prediction pipeline on a graph of scientific papers.
Its goal is to predict whether two papers should have a link (citation).

## Data files

The notebook expects the data files (`abstracts.txt`, `authors.txt`, `edgelist.txt`, `test.txt`) in the same folder.
If the precomputed feature files (`train_features.npz`, `val_features.npz`, `test_features.npz`) are present, they are loaded instead and the data files are not needed.
**Change the paths in the load/save functions if your files are elsewhere.**

In [1]:
!pip install faiss-cpu
!pip install sentence-transformers
!pip install xgboost
import random
import re
import string
from pathlib import Path
from typing import Dict, List, Tuple
import faiss
import numpy as np
import pandas as pd
import torch
import networkx as nx
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from tqdm import tqdm
from xgboost import XGBClassifier
import warnings
from networkx.algorithms.link_prediction import adamic_adar_index

## Data-loading and Text Preprocessing

Loads authors and abstracts from files, cleans the text, and encodes abstracts into embeddings using SentenceTransformer. Normalizes embeddings for later similarity calculations.

In [2]:
# Cleaning function to remove punctuation and convert text to lowercase
def _clean_text(text: str) -> str:
    text = text.lower() # Lowercasing
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text) # Remove punctuation
    return " ".join(text.split())

# Load the authors from the file
def load_authors(file_path: str | Path = "authors.txt") -> Dict[int, set[str]]:
    authors_dict: Dict[int, set[str]] = {} # Create the dict for authors file
    
    with open(file_path, "r", encoding="utf-8") as fh:
        for line in tqdm(fh, desc="Loading Authors"):
            
            if "|--|" not in line:
                continue
            pid, authors = line.strip().split("|--|", 1) #Split based on the delimiter
            
            try:
                pid_int = int(pid)
            except ValueError:
                continue
                
            authors_dict[pid_int] = {a.strip().lower() for a in authors.split(",") if a.strip()} 
    return authors_dict

# Load the abstracts from the file
def load_abstracts(file_path: str | Path = "abstracts.txt") -> Dict[int, str]:
    abstracts: Dict[int, str] = {} # Create the dict for abstracts file
    
    with open(file_path, "r", encoding="utf-8") as fh:
        for line in tqdm(fh, desc="Loading Abstracts"):
            
            if "|--|" not in line:
                continue
            pid, text = line.strip().split("|--|", 1) #Split based on the delimiter
            
            try:
                pid_int = int(pid)
            except ValueError:
                continue
            
            abstracts[pid_int] = _clean_text(text)
    return abstracts

# Preprocess text data to create embeddings
def preprocess_text_data() -> Tuple[Dict[int, set[str]], Dict[int, int], np.ndarray]:
    authors = load_authors() # Load authors
    abstracts = load_abstracts() # Load abstracts

    pids = list(abstracts.keys()) # Get abstracts ids
    texts = [abstracts[pid] for pid in pids] # Get the paper texts

    # Choose the device for embeddings creation (GPU or CPU)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device for embedding:", device)
    model = SentenceTransformer('all-mpnet-base-v2', device=device) # Initialization of the model through SentenceTransformer

    batch_size = 128 
    embeddings_list = []
    
    # Creation of the embeddings of the paper texts 
    for i in tqdm(range(0, len(texts), batch_size), desc="Creating abstracts embeddings"):
        batch_texts = texts[i:i+batch_size]
        emb_batch = model.encode(batch_texts, show_progress_bar=False, convert_to_numpy=True)
        embeddings_list.append(emb_batch)
    embeddings = np.vstack(embeddings_list).astype('float32')

    # Normalization of the embeddings (needed for cosine similarity later)
    faiss.normalize_L2(embeddings)

    return authors, {pid: idx for idx, pid in enumerate(pids)}, embeddings

## Graph and Feature Functions

Loads the graph of papers, generates negative samples and defines feature extraction


In [3]:
# Load the graph from the edgelist file with networkx and convert it to an undirected graph
def load_graph(file_path: str | Path = "edgelist.txt") -> nx.Graph:
    G_dir = nx.read_edgelist(file_path, delimiter=",", create_using=nx.DiGraph(), nodetype=int)
    return G_dir.to_undirected()

# Generate negative samples for link prediction
def generate_negative_samples(G: nx.Graph, num_samples: int) -> List[Tuple[int, int]]:
    nodes = list(G.nodes())
    negatives: set[Tuple[int, int]] = set()
    
    while len(negatives) < num_samples: # Same negative sample size as the positive 
        u, v = random.sample(nodes, 2)
        
        if not G.has_edge(u, v):
            negatives.add((u, v))
    
    return list(negatives)

# Extract features for a given pair of nodes in the graph
def extract_features(pair: Tuple[int, int], G: nx.Graph, authors_dict: Dict[int, set[str]], 
                     pid_to_index: Dict[int, int], embeddings: np.ndarray) -> List[float]:
    
    u, v = pair

    neighbors_u = set(G.neighbors(u)) if G.has_node(u) else set()
    neighbors_v = set(G.neighbors(v)) if G.has_node(v) else set()

    # Jaccard similarity
    common = len(neighbors_u & neighbors_v) # Common neighbors
    union = len(neighbors_u | neighbors_v)
    if union != 0:
        jaccard = common / union
    else:
        jaccard = 0.0

    # Preferential attachment score
    pref_attach = G.degree[u] * G.degree[v] if G.has_node(u) and G.has_node(v) else 0

    # Author overlap (number of shared authors)
    au = authors_dict.get(u, set())
    av = authors_dict.get(v, set())
    common_auth = len(au & av)

    # Cosine similarity
    if u in pid_to_index and v in pid_to_index:
        idx_u = pid_to_index[u]
        idx_v = pid_to_index[v]
        cos = float(np.dot(embeddings[idx_u], embeddings[idx_v])) # Embeddings are normalized, so the dot product is the cosine
    else:
        cos = 0.0

    # Adamic-Adar index
    adamic_adar = 0.0
    if G.has_node(u) and G.has_node(v):
        preds = adamic_adar_index(G, [(u, v)])
        adamic_adar = next(preds)[2]

    return [common, jaccard, pref_attach, common_auth, cos, adamic_adar]


## Training data Pipeline

Prepares the training dataset by combining features of positive and negative links. Saves training and validation sets for future executions of the code.


In [4]:
# Compressed feature files produced by a previous run of the code
TRAIN_FILE = "train_features.npz"
VAL_FILE = "val_features.npz"
TEST_FILE = "test_features.npz"

# Preprocess training data by loading the graph, authors, embeddings and then generating features
def preprocess_training_data():
    print("➤ Preprocessing training data...")
    G = load_graph() # Load the graph
    authors, pid_to_index, embeddings = preprocess_text_data()

    pos_pairs = list(G.edges()) # Get the positive pairs
    neg_pairs = generate_negative_samples(G, len(pos_pairs)) # Get the negative pairs

    X: List[List[float]] = []
    y: List[int] = []

    # Extract features for positive pairs
    for pair in tqdm(pos_pairs, desc="Extracting Positive Features"):
        X.append(extract_features(pair, G, authors, pid_to_index, embeddings))
        y.append(1)

    # Extract features for negative pairs
    for pair in tqdm(neg_pairs, desc="Extracting Negative Features"):
        X.append(extract_features(pair, G, authors, pid_to_index, embeddings))
        y.append(0)

    # Split into train and validation set
    X_train, X_val, y_train, y_val = train_test_split(
        np.array(X, dtype=np.float32), np.array(y), test_size=0.2, random_state=42, stratify=y
    )

    # Save the train and validation sets (compressed) for future executions
    np.savez_compressed(TRAIN_FILE, X=X_train, y=y_train.astype(np.int8))
    np.savez_compressed(VAL_FILE, X=X_val, y=y_val.astype(np.int8))

    return X_train, y_train, X_val, y_val

# Get the training data
def get_training_data():
    if Path(TRAIN_FILE).exists() and Path(VAL_FILE).exists(): # Load the cached features from a previous run
        print("➤ Loading training tensors...")
        train = np.load(TRAIN_FILE)
        val = np.load(VAL_FILE)
        X_train, y_train = train["X"], train["y"]
        X_val, y_val = val["X"], val["y"]
    else: # If the files do not exist, preprocess the training data
        X_train, y_train, X_val, y_val = preprocess_training_data() 
    
    return X_train, y_train, X_val, y_val

## Test data Pipeline

Prepares test dataset pairs for prediction. Extracts the same features as in training and caches the resulting test features.

In [5]:
# Preprocess test data by loading the graph, authors, and embeddings, then extracting features
def preprocess_test_data():
    print("➤ Preprocessing test data...")
    G = load_graph()
    authors, pid_to_index, embeddings = preprocess_text_data()

    # Create a dataframe for test.txt
    df = pd.read_csv("test.txt", header=None, names=["paper1", "paper2"])
    X_test = [extract_features((r.paper1, r.paper2), G, authors, pid_to_index, embeddings) for r in df.itertuples()]
    np.savez_compressed(TEST_FILE, X=np.array(X_test, dtype=np.float32)) # Save the test features for future executions
    
    return np.array(X_test, dtype=np.float32), df

# Get the test data
def get_test_data():
    if Path(TEST_FILE).exists():
        print("➤ Loading test tensors...")
        X_test = np.load(TEST_FILE)["X"]
        df = pd.DataFrame(index=range(len(X_test))) # Only the row index is needed for the submission IDs
    else:
        X_test, df = preprocess_test_data()
    
    return X_test, df

##  Main method and Training Process

Trains an XGBoost model to predict link existence based on extracted features. Evaluates on validation set and produces submission file with predicted probabilities for test pairs.


In [6]:
# Main function to run the training and prediction pipeline
def main():
    X_train, y_train, X_val, y_val = get_training_data()

    # Filter the warnings
    warnings.filterwarnings("ignore", category=UserWarning)
    warnings.filterwarnings("ignore", category=DeprecationWarning)

    # Train on the GPU if one is available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"➤ Training XGBoost on: {device}")

    # Initialize the XGBoost model with hyperparameters
    model = XGBClassifier(
        tree_method="hist",
        device=device,
        eval_metric="logloss", # Log loss metric
        verbosity=0, # Silence XGBoost prints
        learning_rate=0.05, # Set learning rate
        n_estimators=1000, # Set the number of trees
        max_depth=6, # Set the max tree depth 
        subsample=0.8, # 80% sample rate for each tree
        colsample_bytree=0.8, # 80% features rate for each tree
        reg_alpha=0.1, # L1 norm
        reg_lambda=1.0, # L2 norm
        random_state=42, # Seed
        early_stopping_rounds=50 # Early stopping if there is no improvement in the last 50 rounds
    )

    # Train the model
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    print("➤ Training completed!")

    # Calculation of predictions on the validation set
    val_preds_proba = model.predict_proba(X_val)[:, 1]

    # Calculate log loss on the validation set
    val_log_loss = log_loss(y_val, val_preds_proba)
    print(f"➤ Validation Log Loss: {val_log_loss:.6f}")

    # Load of test data
    X_test, df_test = get_test_data()

    # Predictions on the test set
    test_preds_proba = model.predict_proba(X_test)[:, 1]

    # Create submission DataFrame
    submission_df = pd.DataFrame({
        "ID": df_test.index,
        "Label": test_preds_proba
    })

    # Save the submission DataFrame to a CSV file
    submission_df.to_csv("submission.csv", index=False)
    print("➤ Submission file created.")

if __name__ == "__main__":
    main()

➤ Loading training tensors...
➤ Training XGBoost on: cpu
➤ Training completed!
➤ Validation Log Loss: 0.120779
➤ Loading test tensors...
➤ Submission file created.
